<a href="https://colab.research.google.com/github/Adhiaris/UAS-FINALTERM-FOR-DEEP-LEARNING-Text-Summarization/blob/main/Task_3_Text_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3: Text Summarization with Phi-2 on XSum
### NLP Assignment — Decoder-only LLM (Phi-2)


## Step 1: Install Dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate peft bitsandbytes rouge_score -q
print(" All packages installed!")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device : {device}")
if torch.cuda.is_available():
    print(f" GPU : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" VRAM : {vram:.1f} GB")
    if vram < 10:
        print(" → Using LoRA + 4-bit quantization for memory efficiency ")
else:
    print(" No GPU! Phi-2 is very slow on CPU. Enable GPU in Runtime settings.")
print(f" PyTorch: {torch.__version__}")

## Step 2: Load & Explore XSum

In [ ]:
dataset = load_dataset('EdinburghNLP/xsum')
print(dataset)
print("\n Sample entry:")
sample = dataset['train'][0]
print(f" document : {sample['document'][:300]}...")
print(f" summary : {sample['summary']}")
print(f" id : {sample['id']}")

In [ ]:
print(f"Train : {len(dataset['train']):,}")
print(f"Validation: {len(dataset['validation']):,}")
print(f"Test : {len(dataset['test']):,}")

train_sample = dataset['train'].select(range(3000))
doc_lens = [len(ex['document'].split()) for ex in train_sample]
sum_lens = [len(ex['summary'].split()) for ex in train_sample]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(doc_lens, bins=50, color='#3498db', alpha=0.85, edgecolor='white')
axes[0].set_title('Article Length (words)', fontweight='bold')
axes[0].set_xlabel('Words')
axes[0].axvline(400, color='red', linestyle='--', label='400w mark')
axes[0].legend()

axes[1].hist(sum_lens, bins=30, color='#2ecc71', alpha=0.85, edgecolor='white')
axes[1].set_title('Summary Length (words)', fontweight='bold')
axes[1].set_xlabel('Words')

ratios = [d/max(s,1) for d,s in zip(doc_lens, sum_lens)]
axes[2].hist(ratios, bins=40, color='#e74c3c', alpha=0.85, edgecolor='white')
axes[2].set_title('Compression Ratio (article/summary)', fontweight='bold')
axes[2].set_xlabel('Ratio')
axes[2].axvline(np.median(ratios), color='blue', linestyle='--',
                label=f'Median={np.median(ratios):.0f}x')
axes[2].legend()

plt.suptitle('XSum Dataset EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('xsum_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Article length — mean: {np.mean(doc_lens):.0f} | median: {np.median(doc_lens):.0f}")
print(f"Summary length — mean: {np.mean(sum_lens):.1f} | median: {np.median(sum_lens):.0f}")
print(f"Compression — median: {np.median(ratios):.0f}x (XSum is highly abstractive!)")

In [ ]:
print(" XSum Examples (Article → One-sentence Summary):\n")
for i in range(3):
    ex = dataset['train'][i]
    print(f"[{i+1}] Article (first 150 chars): {ex['document'][:150]}...")
    print(f" Summary : {ex['summary']}")
    print()

## Step 3: Load Phi-2 with 4-bit Quantization

In [ ]:
MODEL_NAME = 'microsoft/phi-2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"⏳ Loading {MODEL_NAME} with 4-bit quantization...")
print(" (This may take 1-2 minutes on first run — downloading ~5GB model)")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print(f" Tokenizer loaded | Vocab: {tokenizer.vocab_size:,}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)
model = prepare_model_for_kbit_training(model)

total_params = sum(p.numel() for p in model.parameters())
print(f" Model loaded: {MODEL_NAME}")
print(f" Total parameters: {total_params/1e9:.2f}B")
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    print(f" GPU Memory used : {mem_used:.1f} GB")

## Step 4: LoRA Configuration

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'dense'],
    bias='none'
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(" LoRA applied!")
print(f" Trainable parameters: {trainable:,} ({100*trainable/total:.2f}% of total)")
print(f" Frozen parameters : {total-trainable:,}")
print(f" → Fine-tuning only {trainable/1e6:.1f}M params instead of {total/1e9:.1f}B ")

## Step 5: Preprocessing — Instruction Format

In [ ]:
MAX_LENGTH = 512

def make_prompt(document, summary=None):
    """Create instruction-style prompt for Phi-2."""
    prompt = f"Article: {document}\n\nSummarize the article above in one sentence:\n"
    if summary is not None:
        prompt += summary + tokenizer.eos_token
    return prompt

demo_doc = dataset['train'][0]['document'][:400]
demo_sum = dataset['train'][0]['summary']
print(" Prompt format example:")
print("-" * 60)
print(make_prompt(demo_doc, demo_sum)[:500] + "...")

In [ ]:
def preprocess_xsum(examples):
    prompts = [
        make_prompt(doc[:1000], summ)
        for doc, summ in zip(examples['document'], examples['summary'])
    ]
    tokenized = tokenizer(
        prompts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False
    )
    return tokenized

tokenized = dataset.map(
    preprocess_xsum, batched=True,
    remove_columns=dataset['train'].column_names
)
print("Preprocessing complete!")
print(tokenized)

In [ ]:
USE_SUBSET = True
TRAIN_SIZE = 1000
EVAL_SIZE = 200

if USE_SUBSET:
    train_ds = tokenized['train'].shuffle(seed=42).select(range(TRAIN_SIZE))
    eval_ds = tokenized['validation'].shuffle(seed=42).select(range(EVAL_SIZE))
else:
    train_ds = tokenized['train']
    eval_ds = tokenized['validation']

print(f" Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")

## Step 6: Training

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

total_steps = (TRAIN_SIZE // (4 * 4)) * 2
warmup_steps = int(0.1 * total_steps)

training_args = TrainingArguments(
    output_dir='/content/results/phi2-xsum',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=25,
    report_to='none',
    seed=42,
    fp16=False,
    bf16=torch.cuda.is_available(),
    optim='paged_adamw_8bit',
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)
print("Trainer ready. Starting Phi-2 LoRA fine-tuning...")
print("Expected time: ~15-25 min on T4 GPU")

In [ ]:
train_result = trainer.train()
print("\n Training complete!")
print(f" Loss : {train_result.training_loss:.4f}")
print(f" Runtime: {train_result.metrics['train_runtime']:.0f}s "
      f"({train_result.metrics['train_runtime']/60:.1f} min)")

## Step 7: Evaluation with ROUGE

In [ ]:
rouge = evaluate.load('rouge')

def generate_summary(article, max_new_tokens=60):
    prompt = make_prompt(article[:1000])
    inputs = tokenizer(prompt, return_tensors='pt',
                       max_length=MAX_LENGTH-max_new_tokens,
                       truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    summary = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    for sep in ['.', '\n']:
        if sep in summary:
            summary = summary.split(sep)[0].strip() + ('.' if sep=='.' else '')
            break
    return summary

print("generate_summary() ready. Running ROUGE evaluation on 100 samples...")

In [ ]:
eval_subset = dataset['validation'].select(range(100))
predictions, references = [], []

for i, ex in enumerate(eval_subset):
    pred = generate_summary(ex['document'])
    predictions.append(pred)
    references.append(ex['summary'])
    if (i+1) % 20 == 0:
        print(f" Evaluated {i+1}/100...")

rouge_results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("\n ROUGE Scores (100 validation samples):")
print(f" ROUGE-1 : {rouge_results['rouge1']:.4f}")
print(f" ROUGE-2 : {rouge_results['rouge2']:.4f}")
print(f" ROUGE-L : {rouge_results['rougeL']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

rouge_names = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
rouge_values = [rouge_results['rouge1'], rouge_results['rouge2'], rouge_results['rougeL']]
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = axes[0].bar(rouge_names, rouge_values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('ROUGE Scores', fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim([0, 0.7])
for bar, val in zip(bars, rouge_values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', fontweight='bold')

pred_lens = [len(p.split()) for p in predictions]
ref_lens = [len(r.split()) for r in references]
axes[1].scatter(ref_lens, pred_lens, alpha=0.5, color='steelblue', s=20)
max_len = max(max(pred_lens), max(ref_lens))
axes[1].plot([0, max_len], [0, max_len], 'r--', label='Perfect length match')
axes[1].set_xlabel('Reference Length (words)')
axes[1].set_ylabel('Predicted Length (words)')
axes[1].set_title('Predicted vs Reference Length', fontweight='bold')
axes[1].legend()

logs = trainer.state.log_history
train_loss = [(e['step'], e['loss']) for e in logs if 'loss' in e and 'eval_loss' not in e]
eval_loss = [(e['epoch'], e['eval_loss']) for e in logs if 'eval_loss' in e]
if train_loss:
    steps, losses = zip(*train_loss)
    axes[2].plot(steps, losses, '-', color='#3498db', alpha=0.7, linewidth=1.5, label='Train Loss')
if eval_loss:
    epochs, elosses = zip(*eval_loss)
    ax2b = axes[2].twinx()
    ax2b.plot(epochs, elosses, 'rs-', markersize=8, label='Eval Loss (per epoch)')
    ax2b.set_ylabel('Eval Loss', color='red')
    ax2b.tick_params(axis='y', labelcolor='red')
    ax2b.legend(loc='upper right')
axes[2].set_title('Training Loss Curve', fontweight='bold')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Train Loss')
axes[2].legend(loc='upper left')

plt.suptitle('Phi-2 LoRA Fine-tuning Results on XSum', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('phi2_xsum_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Inference Demo

In [ ]:
print(" Summarization Demo:")
print("=" * 70)
for i in range(5):
    ex = dataset['validation'][i]
    pred = generate_summary(ex['document'])
    print(f"[{i+1}] Article : {ex['document'][:150]}...")
    print(f" Reference: {ex['summary']}")
    print(f" Phi-2 : {pred}")
    print()

In [ ]:
custom_article = """
Scientists have discovered a new species of deep-sea fish in the Mariana Trench,
the deepest part of the world's oceans. The fish, named Pseudoliparis swirei,
was found at a depth of more than 8,000 metres (26,200 feet) below sea level.
The discovery was made by researchers from the University of Aberdeen and the
Chinese Academy of Sciences using remote-operated vehicles. The fish is translucent,
has no scales, and feeds on tiny crustaceans. Researchers say the discovery highlights
how little we know about deep-sea ecosystems and underscores the importance of further
ocean exploration to understand biodiversity in extreme environments.
"""

print(" Custom Article Summarization:")
print("-" * 60)
print(f"Article : {custom_article.strip()[:200]}...")
summary = generate_summary(custom_article)
print(f"Summary : {summary}")

## Step 9: Save Model & Metrics

In [ ]:
SAVE_PATH = '/content/saved_models/phi2-xsum-lora'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

metrics_summary = {
    'model': MODEL_NAME,
    'dataset': 'XSum',
    'task': 'Abstractive Text Summarization',
    'method': 'LoRA fine-tuning (4-bit quantization)',
    'train_samples': len(train_ds),
    'eval_samples': len(eval_ds),
    'rouge1': rouge_results.get('rouge1'),
    'rouge2': rouge_results.get('rouge2'),
    'rougeL': rouge_results.get('rougeL'),
    'train_loss': train_result.training_loss,
    'hyperparams': {
        'epochs': 2,
        'lr': 2e-4,
        'batch_size': 4,
        'gradient_accumulation': 4,
        'effective_batch': 16,
        'max_length': MAX_LENGTH,
        'lora_r': 16,
        'lora_alpha': 32,
        'quantization': '4-bit NF4'
    }
}
with open('phi2_xsum_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print(f" LoRA model saved → {SAVE_PATH}")
print(f" Metrics saved → phi2_xsum_metrics.json")
print()
print(" To load the model later:")
print(" from peft import PeftModel")
print(" model = AutoModelForCausalLM.from_pretrained('microsoft/phi-2', ...)")
print(" model = PeftModel.from_pretrained(model, './phi2-xsum-lora')")